# Parco Veicolare — Tasso di motorizzazione

Analisi del tasso di motorizzazione (autovetture per 1.000 abitanti) in Italia, per macro-area geografica e per i comuni capoluogo, con focus su **Bologna** e **Olbia** nel contesto dell'introduzione della Zona 30.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

In [ ]:
df_raw = pd.read_csv(
    '/content/drive/MyDrive/TesiMagistrale/FontiUsate/parco_veicolare.csv',
    sep=';',
    decimal=',',
    encoding='utf-8'
)

# Teniamo solo l'indicatore CPASCAR1000 e le colonne utili
df = df_raw[df_raw['DATA_TYPE'] == 'CPASCAR1000'][['Territorio', 'TIME_PERIOD', 'Osservazione']].copy()
df = df.rename(columns={'TIME_PERIOD': 'Anno', 'Osservazione': 'Tasso'})
df['Tasso'] = pd.to_numeric(df['Tasso'], errors='coerce')
df['Anno'] = df['Anno'].astype(int)

print(f'✅ Righe caricate: {len(df):,}')
print(f'Anni disponibili: {sorted(df["Anno"].unique())}')

In [ ]:
# Esplora i territori disponibili
print('Territori presenti nel dataset:')
for t in df['Territorio'].unique():
    print(' ', t)

In [ ]:
# Pivot: righe = Anno, colonne = Territorio
pivot = df.pivot_table(index='Anno', columns='Territorio', values='Tasso')
pivot.head()

In [ ]:
# ── Colori coerenti con Zone30.ipynb ────────────────────────────────────
C_BO = '#2166ac'   # blu  — Bologna / Nord-est
C_OL = '#d6604d'   # rosso-arancio — Olbia / Isole
C_IT = '#888888'   # grigio — Italia

# Anni di interesse (adatta in base ai dati disponibili)
anni_disponibili = sorted(df['Anno'].unique())

# ── Nomi macro-aree (verifica con la cella precedente se necessario) ────
MACRO_AREE = {
    'Italia':    {'color': C_IT,      'lw': 2.0, 'ls': '-'},
    'Nord-est':  {'color': C_BO,      'lw': 1.6, 'ls': '--'},
    'Nord-ovest':{'color': '#4393c3', 'lw': 1.6, 'ls': '--'},
    'Centro':    {'color': '#74c476', 'lw': 1.6, 'ls': '--'},
    'Sud':       {'color': '#fd8d3c', 'lw': 1.6, 'ls': '--'},
    'Isole':     {'color': C_OL,      'lw': 1.6, 'ls': '--'},
}

fig, ax = plt.subplots(figsize=(12, 6))

for nome, stile in MACRO_AREE.items():
    if nome in pivot.columns:
        ax.plot(
            pivot.index,
            pivot[nome],
            color=stile['color'],
            lw=stile['lw'],
            ls=stile['ls'],
            label=nome
        )

ax.set_title('Tasso di motorizzazione per macro-area geografica', fontsize=14)
ax.set_xlabel('Anno')
ax.set_ylabel('Autovetture per 1.000 abitanti')
ax.set_xticks(anni_disponibili)
ax.legend(framealpha=0.9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# ── Bologna e Olbia vs macro-area di riferimento e Italia ───────────────
# Bologna → Nord-est; Olbia → Isole
# Adatta i nomi se il dataset usa varianti (es. 'Nord-Est' con maiuscola)

serie = {
    'Italia':   {'col': 'Italia',   'color': C_IT, 'lw': 1.5, 'ls': '-'},
    'Nord-est': {'col': 'Nord-est', 'color': C_BO, 'lw': 1.8, 'ls': '--'},
    'Isole':    {'col': 'Isole',    'color': C_OL, 'lw': 1.8, 'ls': '--'},
    'Bologna':  {'col': 'Bologna',  'color': C_BO, 'lw': 2.5, 'ls': '-'},
    'Olbia':    {'col': 'Olbia',    'color': C_OL, 'lw': 2.5, 'ls': '-'},
}

fig, ax = plt.subplots(figsize=(12, 6))

for label, s in serie.items():
    if s['col'] in pivot.columns:
        ax.plot(
            pivot.index,
            pivot[s['col']],
            color=s['color'],
            lw=s['lw'],
            ls=s['ls'],
            label=label
        )

ax.axvline(x=2023.5, color=C_BO, lw=1.2, ls=':', alpha=0.85, label='Zona 30 Bologna (gen 2024)')
ax.axvline(x=2021.5, color=C_OL, lw=1.2, ls=':', alpha=0.85, label='Zona 30 Olbia (giu 2021)')

ax.set_title('Tasso di motorizzazione: Bologna e Olbia vs macro-area e Italia', fontsize=14)
ax.set_xlabel('Anno')
ax.set_ylabel('Autovetture per 1.000 abitanti')
ax.set_xticks(anni_disponibili)
ax.legend(framealpha=0.9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Tabella riepilogativa: valori per le aree chiave
cols_chiave = [c for c in ['Italia', 'Nord-est', 'Isole', 'Bologna', 'Olbia'] if c in pivot.columns]
pivot[cols_chiave].dropna(how='all')

---
## Età del parco auto

Analisi della composizione per età del parco autovetture. Ogni indicatore rappresenta la **quota percentuale** di autovetture circolanti in quella fascia d'età sul totale del parco.

L'obiettivo è capire se il parco auto si sta **ringiovanendo** a Bologna o nelle Isole (dove si trova Olbia), il che avrebbe implicazioni sulla sicurezza stradale indipendentemente dalle politiche di Zona 30 (le auto più nuove hanno sistemi di sicurezza attiva e passiva più avanzati).

> **Nota su `8YEARCAR`**: questo indicatore è probabilmente il totale aggregato ≥ 8 anni (somma di 8–19, 20–29 e ≥ 30). Viene usato solo nel controllo di coerenza e non viene incluso nei grafici a fasce per evitare doppio conteggio.

In [ ]:
# ── Fasce d'età (6 classi non sovrapposte) ──────────────────────────────
FASCE_ETA = {
    'LES1YEARCAR':   '< 1 anno',
    '1TO3YEARCAR':   '1\u20133 anni',
    '4TO7YEARCAR':   '4\u20137 anni',
    '8TO19YEARCAR':  '8\u201319 anni',
    '20TO29YEARCAR': '20\u201329 anni',
    '30YEARCAR':     '\u226530 anni',
}

df_age = df_raw[df_raw['DATA_TYPE'].isin(FASCE_ETA)][
    ['Territorio', 'DATA_TYPE', 'TIME_PERIOD', 'Osservazione']
].copy()
df_age = df_age.rename(columns={'TIME_PERIOD': 'Anno', 'Osservazione': 'Valore'})
df_age['Valore'] = pd.to_numeric(df_age['Valore'], errors='coerce')
df_age['Anno'] = df_age['Anno'].astype(int)
df_age['Fascia'] = df_age['DATA_TYPE'].map(FASCE_ETA)

print(f'✅ Righe età caricate: {len(df_age):,}')
print(f'Anni disponibili: {sorted(df_age["Anno"].unique())}')

In [ ]:
# ── Controllo di coerenza: le 6 fasce devono sommare ~100 ───────────────
check = (
    df_age
    .groupby(['Territorio', 'Anno'])['Valore']
    .sum()
    .reset_index()
    .rename(columns={'Valore': 'Somma'})
)
print('Somma fasce per Italia (atteso ~100):')
print(check[check['Territorio'] == 'Italia'].to_string(index=False))

In [ ]:
# ── Stacked area chart: composizione per età ────────────────────────────
# Palette verde (giovane) → rosso scuro (anziana)
COLORI_FASCE = {
    '< 1 anno':    '#1a9641',
    '1\u20133 anni':   '#a6d96a',
    '4\u20137 anni':   '#ffffbf',
    '8\u201319 anni':  '#fdae61',
    '20\u201329 anni': '#d7191c',
    '\u226530 anni':   '#7b0000',
}
ORDINE_FASCE = list(COLORI_FASCE.keys())

AREE_STACK = [
    ('Italia',  C_IT,  None),
    ('Bologna', C_BO,  2023.5),
    ('Olbia',   C_OL,  2021.5),
]

fig, axes = plt.subplots(3, 1, figsize=(12, 14), sharex=True)

for ax, (area, col_area, zona30_x) in zip(axes, AREE_STACK):
    df_a = df_age[df_age['Territorio'] == area]
    if df_a.empty:
        ax.set_title(f'{area} — dati non disponibili', fontsize=12)
        continue

    pivot_a = (
        df_a
        .pivot_table(index='Anno', columns='Fascia', values='Valore')
        .reindex(columns=[f for f in ORDINE_FASCE if f in df_a['Fascia'].unique()])
    )
    anni_a = pivot_a.index.tolist()
    colori_a = [COLORI_FASCE[f] for f in pivot_a.columns]

    ax.stackplot(
        anni_a,
        [pivot_a[f].values for f in pivot_a.columns],
        labels=pivot_a.columns.tolist(),
        colors=colori_a,
        alpha=0.85
    )

    if zona30_x is not None:
        ax.axvline(x=zona30_x, color=col_area, lw=1.5, ls=':', alpha=0.9,
                   label=f'Zona 30 {area}')

    ax.set_title(f'Composizione per età del parco auto — {area}', fontsize=12)
    ax.set_ylabel('% autovetture')
    ax.set_ylim(0, 105)
    ax.grid(True, alpha=0.2)
    ax.legend(loc='upper left', fontsize=8, framealpha=0.9)

axes[-1].set_xlabel('Anno')
axes[-1].set_xticks(sorted(df_age['Anno'].unique()))
plt.suptitle('Evoluzione della composizione per età del parco auto (2015–2023)', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ── Quota auto giovani (< 8 anni): LES1 + 1TO3 + 4TO7 ──────────────────
GIOVANI = ['< 1 anno', '1\u20133 anni', '4\u20137 anni']

quota_giovani = (
    df_age[df_age['Fascia'].isin(GIOVANI)]
    .groupby(['Territorio', 'Anno'])['Valore']
    .sum()
    .reset_index()
    .rename(columns={'Valore': 'QuotaGiovani'})
)
piv_g = quota_giovani.pivot_table(index='Anno', columns='Territorio', values='QuotaGiovani')

anni_g = sorted(piv_g.index.tolist())

SERIE_G = [
    ('Italia',   C_IT, 1.5, '-'),
    ('Nord-est', C_BO, 1.8, '--'),
    ('Isole',    C_OL, 1.8, '--'),
    ('Bologna',  C_BO, 2.5, '-'),
    ('Olbia',    C_OL, 2.5, '-'),
]

fig, ax = plt.subplots(figsize=(12, 6))

for label, color, lw, ls in SERIE_G:
    if label in piv_g.columns:
        ax.plot(anni_g, piv_g.loc[anni_g, label], color=color, lw=lw, ls=ls, label=label)

ax.axvline(x=2023.5, color=C_BO, lw=1.2, ls=':', alpha=0.85, label='Zona 30 Bologna (gen 2024)')
ax.axvline(x=2021.5, color=C_OL, lw=1.2, ls=':', alpha=0.85, label='Zona 30 Olbia (giu 2021)')

ax.set_title('Quota di auto con meno di 8 anni sul totale del parco', fontsize=14)
ax.set_xlabel('Anno')
ax.set_ylabel('% autovetture < 8 anni')
ax.set_xticks(anni_g)
ax.yaxis.set_major_formatter(ticker.FormatStrFormatter('%.1f%%'))
ax.legend(framealpha=0.9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# ── Quota auto anziane (≥ 20 anni): 20TO29 + 30YEARCAR ──────────────────
ANZIANE = ['20\u201329 anni', '\u226530 anni']

quota_anziane = (
    df_age[df_age['Fascia'].isin(ANZIANE)]
    .groupby(['Territorio', 'Anno'])['Valore']
    .sum()
    .reset_index()
    .rename(columns={'Valore': 'QuotaAnziane'})
)
piv_a = quota_anziane.pivot_table(index='Anno', columns='Territorio', values='QuotaAnziane')

anni_a = sorted(piv_a.index.tolist())

SERIE_A = [
    ('Italia',   C_IT, 1.5, '-'),
    ('Nord-est', C_BO, 1.8, '--'),
    ('Isole',    C_OL, 1.8, '--'),
    ('Bologna',  C_BO, 2.5, '-'),
    ('Olbia',    C_OL, 2.5, '-'),
]

fig, ax = plt.subplots(figsize=(12, 6))

for label, color, lw, ls in SERIE_A:
    if label in piv_a.columns:
        ax.plot(anni_a, piv_a.loc[anni_a, label], color=color, lw=lw, ls=ls, label=label)

ax.axvline(x=2023.5, color=C_BO, lw=1.2, ls=':', alpha=0.85, label='Zona 30 Bologna (gen 2024)')
ax.axvline(x=2021.5, color=C_OL, lw=1.2, ls=':', alpha=0.85, label='Zona 30 Olbia (giu 2021)')

ax.set_title('Quota di auto con 20 anni o pi\u00f9 sul totale del parco', fontsize=14)
ax.set_xlabel('Anno')
ax.set_ylabel('% autovetture \u226520 anni')
ax.set_xticks(anni_a)
ax.yaxis.set_major_formatter(ticker.FormatStrFormatter('%.1f%%'))
ax.legend(framealpha=0.9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# ── Tabella riepilogativa: quote giovani e anziane per le aree chiave ────
aree_tab = ['Italia', 'Nord-est', 'Isole', 'Bologna', 'Olbia']

cols_g = [c for c in aree_tab if c in piv_g.columns]
cols_a = [c for c in aree_tab if c in piv_a.columns]

tab = pd.concat([
    piv_g[cols_g].add_suffix(' (giovani <8a)'),
    piv_a[cols_a].add_suffix(' (anziane \u226520a)'),
], axis=1).round(1)

tab